# Query Presets & Advanced Retrieval

Neocortex provides several `QueryParam` presets that control the quality/cost/speed tradeoff for queries. This notebook compares them side-by-side and demonstrates advanced retrieval features:

- **Presets**: `balanced()`, `quality()`, `economy()`
- **References mode**: Inline source citations in answers
- **Context-only mode**: Raw scored context without LLM answer generation
- **Custom tuning**: Fine-grained control over ranking thresholds

In [ ]:
import os
import time

from dotenv import load_dotenv

load_dotenv()

from neocortex import GraphRAG, QueryParam
from neocortex._llm import token_tracker

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your .env file"
print("Environment ready.")

## Setup

This notebook reuses the index from `01_quickstart.ipynb`. If you haven't run that notebook yet, the cell below will create a quick index.

In [ ]:
SAMPLE_TEXT = """
Acme Robotics was founded in 2019 by Dr. Elena Vasquez and Marcus Chen in Austin, Texas.
The company specializes in autonomous warehouse robots and has grown to 120 employees.
Their flagship product, the AcmeBot X1, uses LiDAR and computer vision to navigate
complex warehouse environments. It can carry up to 50 kg and operates for 12 hours on
a single charge.

Dr. Vasquez serves as CEO and leads the research division. She holds a PhD in robotics
from MIT and previously worked at Boston Dynamics for eight years, where she led the
warehouse automation team. Marcus Chen is the CTO and oversees all engineering. He was
formerly a senior engineer at Tesla's Gigafactory, specializing in robotic assembly lines.

In 2022, Acme Robotics raised a $45 million Series B round led by Sequoia Capital, with
participation from Andreessen Horowitz and Y Combinator. The funding was used to expand
their Austin headquarters and open a new R&D lab in San Francisco.

The company's biggest client is GlobalMart, a retail giant that has deployed 200 AcmeBot
X1 units across its distribution centers in Dallas, Chicago, and Atlanta. GlobalMart
reported a 35% increase in warehouse efficiency after adopting the AcmeBot system.

In early 2024, Acme announced the AcmeBot X2, featuring improved AI navigation, a
75 kg payload capacity, and swarm coordination capabilities that allow multiple robots
to work together seamlessly. The X2 is being piloted by FreshFoods Inc., a grocery
logistics company based in Portland, Oregon.

Dr. Vasquez recently published a paper in the IEEE Robotics journal titled "Scalable
Multi-Agent Navigation in Dense Warehouse Environments" co-authored with Dr. James
Park, the company's Head of AI Research. Dr. Park joined Acme from Google DeepMind
in 2021 and leads a team of 15 machine learning engineers.

Acme Robotics holds 12 patents related to autonomous navigation and has filed 5 more
in 2024 covering swarm intelligence algorithms. The company is headquartered at 500
Innovation Drive, Austin, TX 78701.
""".strip()

rag = GraphRAG(
    working_dir="./db/examples_quickstart",
    domain="information about a robotics company, its people, products, investors, and clients",
    example_queries=(
        "Who founded Acme Robotics?\n"
        "What products does the company make?\n"
        "Who are the major investors?"
    ),
    entity_types=["person", "company", "product", "location", "organization"],
)

# Try loading existing index; if it doesn't exist, create one
from pathlib import Path

if not Path("./db/examples_quickstart").exists():
    print("No existing index found. Indexing sample text...")
    token_tracker.reset()
    rag.insert(SAMPLE_TEXT)
    print("Indexing complete.")
else:
    print("Using existing index from ./db/examples_quickstart")

## Compare Presets: Balanced vs Quality vs Economy

Each preset tunes the retrieval pipeline differently:

| Preset | Entity Threshold | Relation Top-K | Chunk Top-K | Token Budgets |
|--------|-----------------|----------------|-------------|---------------|
| `balanced()` | 0.005 | 64 | 8 | Standard |
| `quality()` | 0.001 | 128 | 16 | Standard |
| `economy()` | 0.01 | 32 | 4 | Reduced |

Let's run the same question with each preset and compare.

In [ ]:
QUESTION = "What are all the products Acme Robotics makes and who are their clients?"

presets = {
    "balanced": QueryParam.balanced(),
    "quality": QueryParam.quality(),
    "economy": QueryParam.economy(),
}

comparison = []

for name, param in presets.items():
    token_tracker.reset()
    start = time.perf_counter()

    response = rag.query(QUESTION, params=param)

    elapsed = time.perf_counter() - start
    cost = token_tracker.total_cost

    comparison.append({
        "preset": name,
        "latency": elapsed,
        "cost": cost,
        "entities": len(response.context.entities),
        "relations": len(response.context.relations),
        "chunks": len(response.context.chunks),
        "answer_len": len(response.response),
        "answer": response.response,
    })

    print(f"\n{'=' * 60}")
    print(f"Preset: {name.upper()}")
    print(f"{'=' * 60}")
    print(f"Latency: {elapsed:.1f}s | Cost: ${cost:.4f}")
    print(f"Context: {len(response.context.entities)} entities, "
          f"{len(response.context.relations)} relations, "
          f"{len(response.context.chunks)} chunks")
    print(f"\nAnswer:\n{response.response}")

In [ ]:
# Comparison table
print(f"{'Preset':<10} {'Latency':>8} {'Cost':>8} {'Entities':>9} {'Relations':>10} {'Chunks':>7} {'Answer Len':>11}")
print("-" * 70)
for c in comparison:
    print(f"{c['preset']:<10} {c['latency']:>7.1f}s ${c['cost']:>6.4f} {c['entities']:>9} "
          f"{c['relations']:>10} {c['chunks']:>7} {c['answer_len']:>11}")

## References Mode

Setting `with_references=True` asks the LLM to include inline source citations `[1]`, `[2]`, etc. in its answer, referencing the retrieved chunks.

In [ ]:
token_tracker.reset()
start = time.perf_counter()

response = rag.query(
    "What funding has Acme Robotics received and from which investors?",
    params=QueryParam(with_references=True),
)

elapsed = time.perf_counter() - start
print(f"Query completed in {elapsed:.1f}s\n")
print("Answer with references:")
print(response.response)
print()

# Show the source chunks that the references point to
print("\nSource chunks:")
for i, (chunk, score) in enumerate(response.context.chunks):
    preview = chunk.content[:150].replace("\n", " ")
    print(f"  [{i+1}] (score: {score:.4f}) {preview}...")

## Context-Only Mode

Setting `only_context=True` skips the LLM answer generation and returns just the scored context. This is useful when you want to:
- Build your own answer generation pipeline
- Use the context with a different LLM
- Inspect retrieval quality without paying for answer generation

In [ ]:
token_tracker.reset()
start = time.perf_counter()

response = rag.query(
    "Tell me about Dr. James Park",
    params=QueryParam(only_context=True),
)

elapsed = time.perf_counter() - start
print(f"Context retrieved in {elapsed:.1f}s (${token_tracker.total_cost:.4f})")
print(f"Response text: '{response.response}'  (empty — no LLM answer generated)\n")

print("Scored Entities:")
for entity, score in response.context.entities:
    print(f"  {score:.4f}  [{entity.type}] {entity.name}: {entity.description[:80]}")

print("\nScored Relations:")
for relation, score in response.context.relations:
    print(f"  {score:.4f}  {relation.source} --[{relation.rel_type}]--> {relation.target}")

print("\nScored Chunks:")
for chunk, score in response.context.chunks:
    preview = chunk.content[:120].replace("\n", " ")
    print(f"  {score:.4f}  {preview}...")

## Custom QueryParam Tuning

For fine-grained control, you can override individual ranking parameters. This is useful when you know your graph characteristics and want to optimize retrieval.

In [ ]:
# Wide retrieval: low threshold + high top-k to cast a wide net
wide_params = QueryParam(
    entity_ranking_threshold=0.001,  # Include more entities
    relation_ranking_top_k=128,      # More relations
    chunk_ranking_top_k=16,          # More chunks
    entities_max_tokens=6000,        # Larger token budget for entity context
    relations_max_tokens=5000,
    chunks_max_tokens=12000,
)

# Narrow retrieval: high threshold + low top-k for precision
narrow_params = QueryParam(
    entity_ranking_threshold=0.05,   # Only top entities
    relation_ranking_top_k=16,       # Few relations
    chunk_ranking_top_k=2,           # Minimal chunks
    entities_max_tokens=2000,
    relations_max_tokens=1000,
    chunks_max_tokens=3000,
)

QUESTION = "What technology does Acme Robotics use in its products?"

for label, params in [("wide", wide_params), ("narrow", narrow_params)]:
    token_tracker.reset()
    start = time.perf_counter()

    response = rag.query(QUESTION, params=params)

    elapsed = time.perf_counter() - start

    print(f"\n--- {label.upper()} retrieval ---")
    print(f"Latency: {elapsed:.1f}s | Cost: ${token_tracker.total_cost:.4f}")
    print(f"Context: {len(response.context.entities)} entities, "
          f"{len(response.context.relations)} relations, "
          f"{len(response.context.chunks)} chunks")
    print(f"Answer: {response.response[:300]}")

## Guidelines for Choosing Presets

| Use Case | Recommended Preset | Why |
|----------|-------------------|-----|
| General Q&A | `balanced()` | Good quality at reasonable cost |
| High-stakes analysis | `quality()` | Maximum context, best answers |
| High-volume / batch queries | `economy()` | Lowest cost per query |
| Source attribution | `QueryParam(with_references=True)` | Inline citations |
| Custom pipelines | `QueryParam(only_context=True)` | BYO answer generation |
| Domain-specific tuning | Custom `QueryParam(...)` | Full control over retrieval |

**Next steps:**
- [05_multi_source_knowledge_base.ipynb](05_multi_source_knowledge_base.ipynb) — Incremental indexing and workspace isolation
- [06_graph_visualization.ipynb](06_graph_visualization.ipynb) — Export and visualize the knowledge graph